In [4]:
import random
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

REPO_ROOT = Path.cwd().parents[1]          # examples/my_ac_video_jepa -> repo root
sys.path.insert(0, str(REPO_ROOT))

from eb_jepa.datasets.utils import init_data
from eb_jepa.datasets.two_rooms.utils import generate_wall_layouts

torch.manual_seed(0); np.random.seed(0); random.seed(0)

### `load_env_data_config`

In [ ]:
import yaml 


DATASETS_DIR = Path("/Users/hawardizayee/Desktop/AMI/eb_jepa/eb_jepa/datasets")
config_path = DATASETS_DIR / "two_rooms" / "data_config.yaml"

print(config_path)

/Users/hawardizayee/Desktop/AMI/eb_jepa/eb_jepa/datasets/two_rooms/data_config.yaml


In [8]:
with open(config_path) as f:
    base_config = yaml.safe_load(f)
    

In [9]:
base_config

{'action_noise': 1,
 'action_angle_noise': 0.2,
 'action_step_mean': 1.0,
 'action_step_std': 0.4,
 'action_lower_bd': 0.2,
 'action_upper_bd': 1.8,
 'dot_std': 1.3,
 'img_size': 65,
 'border_wall_loc': 5,
 'fix_wall_batch_k': None,
 'fix_wall': False,
 'fix_door_location': 18,
 'fix_wall_location': 32,
 'exclude_wall_train': '',
 'exclude_door_train': '',
 'only_wall_val': '',
 'only_door_val': '',
 'wall_padding': 20,
 'door_padding': 10,
 'wall_width': 3,
 'door_space': 4,
 'num_train_layouts': -1,
 'cross_wall_rate': 0.35,
 'expert_cross_wall_rate': 0,
 'wall_bump_rate': 0.0,
 'dup_traj_rate': 0.0,
 'max_step': 1,
 'sample_length': 17,
 'n_steps': 91,
 'n_steps_reduce_factor': 1,
 'repeat_actions': 1,
 'size': 100000,
 'val_size': 10000,
 'train': True,
 'normalize': True,
 'batch_size': 64,
 'num_workers': 0,
 'pin_mem': False,
 'persistent_workers': False,
 'device': 'cpu'}

In [10]:
overrides = {"batch_size": 128}
base_config.update(overrides)

In [11]:
base_config

{'action_noise': 1,
 'action_angle_noise': 0.2,
 'action_step_mean': 1.0,
 'action_step_std': 0.4,
 'action_lower_bd': 0.2,
 'action_upper_bd': 1.8,
 'dot_std': 1.3,
 'img_size': 65,
 'border_wall_loc': 5,
 'fix_wall_batch_k': None,
 'fix_wall': False,
 'fix_door_location': 18,
 'fix_wall_location': 32,
 'exclude_wall_train': '',
 'exclude_door_train': '',
 'only_wall_val': '',
 'only_door_val': '',
 'wall_padding': 20,
 'door_padding': 10,
 'wall_width': 3,
 'door_space': 4,
 'num_train_layouts': -1,
 'cross_wall_rate': 0.35,
 'expert_cross_wall_rate': 0,
 'wall_bump_rate': 0.0,
 'dup_traj_rate': 0.0,
 'max_step': 1,
 'sample_length': 17,
 'n_steps': 91,
 'n_steps_reduce_factor': 1,
 'repeat_actions': 1,
 'size': 100000,
 'val_size': 10000,
 'train': True,
 'normalize': True,
 'batch_size': 128,
 'num_workers': 0,
 'pin_mem': False,
 'persistent_workers': False,
 'device': 'cpu'}

### `update_config_from_yaml`

In [14]:
from dataclasses import fields
from eb_jepa.datasets.two_rooms.wall_dataset import WallDatasetConfig

In [ ]:
config_class = WallDatasetConfig
fields(config_class)

(Field(name='size',type=<class 'int'>,default=10000,default_factory=<dataclasses._MISSING_TYPE object at 0x10ab98320>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,_field_type=_FIELD),
 Field(name='val_size',type=<class 'int'>,default=10000,default_factory=<dataclasses._MISSING_TYPE object at 0x10ab98320>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,_field_type=_FIELD),
 Field(name='batch_size',type=<class 'int'>,default=128,default_factory=<dataclasses._MISSING_TYPE object at 0x10ab98320>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,_field_type=_FIELD),
 Field(name='dot_std',type=<class 'float'>,default=1.3,default_factory=<dataclasses._MISSING_TYPE object at 0x10ab98320>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),kw_only=False,_field_type=_FIELD),
 Field(name='action_noise',type=<class 'float'>,default=0.2,default_factory=<dataclasses._MISSING_TYP

In [17]:
config_field_names = {f.name for f in fields(config_class)}
config_field_names

{'action_angle_noise',
 'action_lower_bd',
 'action_noise',
 'action_step_mean',
 'action_step_std',
 'action_upper_bd',
 'batch_size',
 'border_wall_loc',
 'chunked_actions',
 'cross_wall_rate',
 'device',
 'door_padding',
 'door_space',
 'dot_std',
 'dup_traj_rate',
 'exclude_door_train',
 'exclude_wall_train',
 'expert_action_lower_bd',
 'expert_action_step_mean',
 'expert_action_step_std',
 'expert_action_upper_bd',
 'expert_cross_wall_rate',
 'expert_traj_door_padding',
 'fix_door_location',
 'fix_wall',
 'fix_wall_batch_k',
 'fix_wall_location',
 'image_based',
 'img_size',
 'max_step',
 'n_steps',
 'n_steps_reduce_factor',
 'normalize',
 'num_train_layouts',
 'only_door_val',
 'only_wall_val',
 'repeat_actions',
 'sample_length',
 'size',
 'train',
 'val_size',
 'wall_bump_rate',
 'wall_padding',
 'wall_width'}

In [18]:
relevant_yaml_data = {
    key: value for key, value in base_config.items() if key in config_field_names
}

relevant_yaml_data

{'action_noise': 1,
 'action_angle_noise': 0.2,
 'action_step_mean': 1.0,
 'action_step_std': 0.4,
 'action_lower_bd': 0.2,
 'action_upper_bd': 1.8,
 'dot_std': 1.3,
 'img_size': 65,
 'border_wall_loc': 5,
 'fix_wall_batch_k': None,
 'fix_wall': False,
 'fix_door_location': 18,
 'fix_wall_location': 32,
 'exclude_wall_train': '',
 'exclude_door_train': '',
 'only_wall_val': '',
 'only_door_val': '',
 'wall_padding': 20,
 'door_padding': 10,
 'wall_width': 3,
 'door_space': 4,
 'num_train_layouts': -1,
 'cross_wall_rate': 0.35,
 'expert_cross_wall_rate': 0,
 'wall_bump_rate': 0.0,
 'dup_traj_rate': 0.0,
 'max_step': 1,
 'sample_length': 17,
 'n_steps': 91,
 'n_steps_reduce_factor': 1,
 'repeat_actions': 1,
 'size': 100000,
 'val_size': 10000,
 'train': True,
 'normalize': True,
 'batch_size': 128,
 'device': 'cpu'}

In [ ]:
config_class(**relevant_yaml_data)  # notice the batch_size has overridden 

WallDatasetConfig(size=100000, val_size=10000, batch_size=128, dot_std=1.3, action_noise=1, action_angle_noise=0.2, action_step_mean=1.0, action_step_std=0.4, action_lower_bd=0.2, action_upper_bd=1.8, max_step=1, n_steps=91, img_size=65, train=True, device='cpu', repeat_actions=1, n_steps_reduce_factor=1, border_wall_loc=5, chunked_actions=False, normalize=True, fix_wall=False, fix_wall_batch_k=None, wall_padding=20, door_padding=10, wall_width=3, door_space=4, cross_wall_rate=0.35, expert_cross_wall_rate=0, wall_bump_rate=0.0, dup_traj_rate=0.0, expert_action_step_mean=0.9, expert_action_step_std=0, expert_action_lower_bd=0.9, expert_action_upper_bd=0.9, expert_traj_door_padding=2, exclude_wall_train='', exclude_door_train='', only_wall_val='', only_door_val='', fix_wall_location=32, fix_door_location=18, num_train_layouts=-1, image_based=True, sample_length=17)